# Phase 14: Source Manager Test
This notebook verifies adding, removing, and deduplicating sources across FAISS, SQLite, and Graph storage.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.storage.faiss_store import FAISSStore
from src.storage.sqlite_manager import SQLiteManager
from src.graph.graph_storage import GraphStorage
from src.storage.source_manager import SourceManager
print("Import successful!")

## 1. Test Source Manager Lifecycle

In [ ]:
import shutil
# Cleanup past tests
for path in ["test_src_faiss", "test_src_meta.db", "test_src_graph"]:
    if os.path.exists(path):
        if os.path.isdir(path):
            shutil.rmtree(path)
        else:
            os.remove(path)

faiss = FAISSStore(dimension=3, index_path="test_src_faiss/index.faiss")
sqlite = SQLiteManager(db_path="test_src_meta.db")
graph = GraphStorage(graph_path="test_src_graph/graph.pkl")
manager = SourceManager(faiss_store=faiss, sqlite_manager=sqlite, graph_storage=graph)

# 1. Add Source
source_data = {"title": "Test PDF", "source_type": "pdf"}
chunks = [
    {"id": "c1", "content": "Hello world", "embedding": [1.0, 0.0, 0.0]},
    {"id": "c2", "content": "Testing", "embedding": [0.0, 1.0, 0.0]}
]

source_id = manager.add_source(source_data, chunks)
print(f"Added source: {source_id}")
assert len(manager.get_all_sources()) == 1
assert faiss.get_stats()["total_chunks"] == 2

# 2. Test Deduplication
chunks_dup = [
    {"id": "c3", "content": "Hello world", "embedding": [1.0, 0.0, 0.0]} # Duplicate content
]
manager.add_source({"title": "Dup PDF", "source_type": "pdf"}, chunks_dup)
removed = manager.remove_duplicates()
print(f"Removed duplicates: {removed}")
assert removed == 1

# 3. Remove Source
success = manager.remove_source(source_id)
print(f"Remove success: {success}")
assert success == True
assert len(manager.get_all_sources()) == 1 # Dup PDF remains

print("Source Manager test passed!")